# Part D: Full RAG Pipeline Implementation
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015

This notebook integrates the work from Parts A, B, and C into a single, cohesive pipeline with full logging at each stage.

### 1. Setup and Initialization

In [ ]:
import json
import faiss
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

# Load Assets
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks

model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('indexes/rag_index.faiss')

texts = [c['text'] for c in all_chunks]
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ Pipeline components initialized successfully.")

### 2. The RAG Pipeline Function

This function connects all the stages and includes logging for transparency.

In [ ]:
def rag_pipeline(query, k=3, verbose=True):
    """
    Full RAG Pipeline:
    Query -> Retrieval -> Context Selection -> Prompt -> LLM -> Response
    """
    if verbose: print(f"[LOG] Starting pipeline for query: '{query}'")
    
    # --- STAGE 1: RETRIEVAL ---
    if verbose: print("[LOG] Stage 1: Retrieving relevant documents...")
    
    # Vector Search
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    
    # Keyword Search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    # Combine for Hybrid results
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        score = (0.5 * v_score) + (0.5 * (bm25_scores[i] / max_bm25))
        if score > 0: combined.append({'chunk': chunk, 'score': score})
    
    results = sorted(combined, key=lambda x: x['score'], reverse=True)[:k]
    
    if verbose:
        print(f"[LOG] Found {len(results)} relevant chunks.")
        for i, res in enumerate(results):
            print(f"      - Chunk {i+1} | Score: {res['score']:.4f} | Source: {res['chunk']['source']}")
    
    # --- STAGE 2: CONTEXT SELECTION ---
    if verbose: print("[LOG] Stage 2: Building context block...")
    context = ""
    for res in results:
        context += f"\n[Source: {res['chunk']['source']}] {res['chunk']['text']}\n"
    
    # --- STAGE 3: PROMPT CONSTRUCTION ---
    if verbose: print("[LOG] Stage 3: Constructing final prompt...")
    final_prompt = f"""Answer the question based only on the provided documents.
If the answer is not in the context, say you do not know.

Documents:
{context}

User Query: {query}
Answer:"""
    
    if verbose:
        print("--- FINAL PROMPT SENT TO LLM ---")
        print(final_prompt)
        print("--------------------------------")
    
    # --- STAGE 4: LLM GENERATION ---
    if verbose: print("[LOG] Stage 4: Querying Groq (Llama-3)...")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    
    answer = response.choices[0].message.content
    if verbose: print("[LOG] Pipeline complete.")
    
    return answer

### 3. Pipeline Execution and Demonstration

In [ ]:
query = "What is the inflation target for 2025?"
final_answer = rag_pipeline(query)

print("\n--- FINAL RESPONSE ---")
print(final_answer)